# MoP+DivPO — SFT Training (Google Colab)

Trains one LoRA adapter per persona on a T4 GPU (~15 min per persona at 200 steps).

**Before running this notebook**, push training data from your local machine:
```bash
source .venv/bin/activate
python -m mop_divpo.training.push_data
```

Trained adapters are pushed to HuggingFace Hub automatically after each persona.

In [1]:
# source .venv/bin/activate
# python -m mop_divpo.data.acquire        # download raw datasets
# python -m mop_divpo.data.prepare        # build SFT JSONL per persona → data/processed/sft/
# python -m mop_divpo.training.push_data  # push to HF: DasonTio/mop-divpo-sft-data

In [9]:
# Cell 1 — Install dependencies
!pip install -q "trl==1.3.0" "peft>=0.16.0" transformers datasets accelerate huggingface_hub pydantic pyyaml "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.7 MB/s eta 0:00:00


In [3]:
# Cell 2 — Check GPU
import torch

assert torch.cuda.is_available(), "No GPU detected. Set Runtime > Change runtime type > T4 GPU"
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# Cell 3 — HuggingFace login (required to push trained adapters)
from huggingface_hub import login
login()  # paste your HF write-access token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# Cell 4 — Load SFT training data from HuggingFace Hub
import json
from pathlib import Path
from datasets import load_dataset

HF_DATASET = 'DasonTio/mop-divpo-sft-data'
DATA_DIR = Path('/content/sft_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset(HF_DATASET)
for persona, split in ds.items():
    out = DATA_DIR / f'{persona}.jsonl'
    with out.open('w') as f:
        for row in split:
            f.write(json.dumps(row) + '\n')
    print(f'{persona}: {len(split)} records saved to {out}')

README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

data/contrarian-00000-of-00001.parquet:   0%|          | 0.00/835k [00:00<?, ?B/s]

data/cross_domain_analogist-00000-of-000(…):   0%|          | 0.00/293k [00:00<?, ?B/s]

data/systems_thinker-00000-of-00001.parq(…):   0%|          | 0.00/749k [00:00<?, ?B/s]

data/minimalist-00000-of-00001.parquet:   0%|          | 0.00/107k [00:00<?, ?B/s]

Generating contrarian split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating cross_domain_analogist split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating systems_thinker split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating minimalist split:   0%|          | 0/352 [00:00<?, ? examples/s]

contrarian: 499 records saved to /content/sft_data/contrarian.jsonl
cross_domain_analogist: 500 records saved to /content/sft_data/cross_domain_analogist.jsonl
systems_thinker: 500 records saved to /content/sft_data/systems_thinker.jsonl
minimalist: 352 records saved to /content/sft_data/minimalist.jsonl


In [6]:
# Cell 5 — Training utilities (self-contained, no GitHub clone needed)
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

PERSONA_IDS = ('contrarian', 'cross_domain_analogist', 'systems_thinker', 'minimalist')
PERSONA_DESCRIPTIONS = {
    'contrarian': 'Challenges assumptions, surfaces minority dissent, and reframes common viewpoints.',
    'cross_domain_analogist': 'Builds analogies across distant fields to transfer useful structures.',
    'systems_thinker': 'Explains causal relationships, feedback loops, trade-offs, and long-term effects.',
    'minimalist': 'Uses constraints, compression, and removal of non-essential parts to sharpen ideas.',
}
HF_ADAPTER_PREFIX = 'DasonTio/mop-divpo'
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def format_records(records, tokenizer, persona_description):
    formatted = []
    for r in records:
        prompt = str(r.get('prompt', '')).strip()
        response = str(r.get('response', '')).strip()
        if not prompt or not response:
            continue
        messages = [
            {'role': 'system', 'content': persona_description},
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': response},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted.append({'text': text})
    return formatted


def train_persona(
    persona,
    data_path,
    output_dir,
    max_steps=200,
    per_device_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lora_r=16,
    lora_alpha=32,
    push_to_hub=True,
):
    assert persona in PERSONA_IDS, f'Unknown persona: {persona}'
    description = PERSONA_DESCRIPTIONS[persona]
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f'Loading tokenizer from {BASE_MODEL}...')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    records = read_jsonl(data_path)
    formatted = format_records(records, tokenizer, description)
    assert formatted, f'No valid records in {data_path}'
    print(f'Training records: {len(formatted)}')

    dataset = Dataset.from_list(formatted)

    print('Loading base model...')
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
    )

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj'],
        bias='none',
    )

    sft_config = SFTConfig(
        output_dir=str(output_dir),
        max_steps=max_steps,
        per_device_train_batch_size=per_device_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        fp16=True,
        bf16=False,
        logging_steps=10,
        save_steps=max_steps,
        save_total_limit=1,
        dataloader_pin_memory=True,
        report_to='none',
        dataset_text_field='text',
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=dataset,
        peft_config=lora_config,
    )

    print(f'Training {persona}...')
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f'Adapter saved to {output_dir}')

    if push_to_hub:
        repo_id = f'{HF_ADAPTER_PREFIX}-{persona}-sft'
        print(f'Pushing to {repo_id}...')
        trainer.model.push_to_hub(repo_id)
        tokenizer.push_to_hub(repo_id)
        print(f'Done: https://huggingface.co/{repo_id}')

    # Free VRAM before next persona
    del model, trainer
    torch.cuda.empty_cache()


print('Training utilities ready.')

Training utilities ready.


In [10]:
# Cell 6 — Train all personas
DATA_DIR = Path('/content/sft_data')
OUTPUT_DIR = Path('/content/adapters')
MAX_STEPS = 200

for persona in PERSONA_IDS:
    data_path = DATA_DIR / f'{persona}.jsonl'
    if not data_path.exists():
        print(f'Skipping {persona}: {data_path} not found')
        continue
    print(f'\n=== {persona} ===')
    train_persona(
        persona=persona,
        data_path=data_path,
        output_dir=OUTPUT_DIR / persona,
        max_steps=MAX_STEPS,
        push_to_hub=True,
    )

print('\nAll personas trained.')


=== contrarian ===
Loading tokenizer from Qwen/Qwen2.5-0.5B-Instruct...
Training records: 499
Loading base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/499 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpeel6qprd/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Done: https://huggingface.co/DasonTio/mop-divpo-minimalist-sft

All personas trained.


In [16]:
# Cell 7 — Smoke test: load a trained adapter and generate
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# contrarian, cross_domain_analogist, minimalist, systems_thinker
PERSONA = 'minimalist'  # change to test other personas
ADAPTER_REPO = f'DasonTio/mop-divpo-{PERSONA}-sft'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map='auto'
)
model = PeftModel.from_pretrained(model, ADAPTER_REPO)
model.eval()

prompt = 'Help me to decide a title for medium article. Give me the reasoning about it too'
messages = [
    {'role': 'system', 'content': PERSONA_DESCRIPTIONS[PERSONA]},
    {'role': 'user', 'content': prompt},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'[{PERSONA}]\n{response}')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.34M [00:00<?, ?B/s]

[minimalist]
Medium articles are often used as a kind of advertisement, and there is much that is
inherent in their nature which might be omitted from any other description.
They contain what is called a “tippet,” or summary, of all the principal points
and authorities referred to, which are required by law. The main point is,
therefore, to ascertain the best form of publication they can take; so far as
that question depends on the circumstances of the particular case, no one but
the author will do justice to himself if he attempts to devise something else. In
this way the public are enabled to choose, not among the books already published,
but among those yet to be produced, the best ones for the purpose. If the
publication of the article must follow the practice adopted by the publisher, it
is desirable to avoid doing so at the expense of any other work in the series;
and, also, where this is possible, to show what advantages are obtained by the use
of these medium articles. Thus if 